In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fcustom_satellite_embeddings%2Fnotebooks%2Fee_data_exploration%2FCustom_Satellite_Embeddings_Data_Access.ipynb?utm_source=custom_satellite_embeddings_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
</table>

# Custom Satellite Embeddings Data Access

Custom satellite embeddings imagery is delivered as Cloud Optimized Geotiffs (COGs) in Google Cloud Storage. COGs in Cloud Storage can be loaded into Google Earth Engine for visualization and analysis. They can be loaded directly using [`ee.Image.loadGeoTIFF()`](https://developers.google.com/earth-engine/apidocs/ee-image-loadgeotiff). Or you can create Earth Engine `Image` and `ImageCollection` assets backed by COGs in Cloud Storage using [`ee.data.createAsset()`](https://developers.google.com/earth-engine/apidocs/ee-data-createasset) and [`ee.data.startExternalImageIngestion()`](https://github.com/google/earthengine-api/blob/master/python/ee/data.py#L1967). The advantage of creating COG-backed assets is that indices on spatial and metadata properties make collections of COG-backed assets performant. The COGs used in this guide are public GeoTIFFs stored in a demo dataset bucket.

This guide demonstrates:

- How to create COG-backed collections for further exploration or analysis in Earth Engine using `ee.data` client operations.
- How to dynamically load COG-backed collections using `ee.Image.loadGeoTIFF`.

## Earth Engine Authentication and Initialization

To use Earth Engine, you must have a Cloud Project with the Earth Engine API activated.

In [ ]:
from datetime import datetime, timezone
import ee
import geemap
import json
from google.cloud import storage
from urllib.parse import urlparse

In [ ]:
# Your project with the Earth Engine API activated.
PROJECT = 'YOUR-PROJECT'

In [ ]:
# Initialize Earth Engine (Assumes you have already authenticated via gcloud)
try:
  ee.Initialize(project=PROJECT)
except Exception:
  ee.Authenticate()
  ee.Initialize(project=PROJECT)

## Make COG-backed image collections

You can store COG-backed images and image collections in your own folders for ease of access.  The advantage of doing this is that you can load your created `ImageCollection` without any further setup.  This guide demonstrates [`ee.data.createAsset()`](https://developers.google.com/earth-engine/apidocs/ee-data-createasset) to create the `ImageCollection` and [`ee.data.startExternalImageIngestion()`](https://github.com/google/earthengine-api/blob/master/python/ee/data.py#L1967) to create the COG-backed `Image` assets in the `ImageCollection`.  

For dynamically created image collections, see the section below.  See also the [Cloud GeoTIFF backed assets](https://developers.google.com/earth-engine/Earth_Engine_asset_from_cloud_geotiff) guide, which shows how to use the [Earth Engine command line](https://developers.google.com/earth-engine/guides/command_line) or [REST API](https://developers.google.com/earth-engine/reference/rest) to achieve the same result.

In [ ]:
# REPLACE THE BELOW WITH YOUR OWN BUCKET/FOLDER/DATASET paths.

# The Google Cloud Storage bucket storing one or more folders.
bucket='alphaearth_foundations_custom'
# A folder storing one or more datasets.
datasets_folder = 'custom_satellite_embedding/v1'
# A folder in datasets_folder storing a dataset. Also the ImageCollection name.
dataset = 'meloland_california_2026_crop_cycles'
# The complete complete GCS URI.
gcs_uri = f'gs://{bucket}/{datasets_folder}/{dataset}/'

print(f'Data will be loaded from: {gcs_uri}')

### Create an `ImageCollection` to contain the images

Check if an `ImageCollection` with the dataset name already exists in the project. If not, create one. If the collections exists, images will be added to it.

In [ ]:
new_image_collection_asset_id = f'projects/{PROJECT}/assets/{dataset}'

try:
  ee.data.getAsset(new_image_collection_asset_id)
except ee.EEException:
  print(f'Creating image collection: {new_image_collection_asset_id}')
  ee.data.createAsset({
      'type': 'IMAGE_COLLECTION', 'name': new_image_collection_asset_id
  })

### List the COG manifests

The subdirectories of the dataset folder contain `image_manifest.json` files which can be used to create the COG-backed assets directly ([example](https://storage.mtls.cloud.google.com/alphaearth_foundations_custom/custom_satellite_embedding/v1/meloland_california_2026_crop_cycles/20251115_20251215/xefjenvdw5qcaxr6w-0000000000-0000008192/image_manifest.json)). Make a list of all the manifests using the Cloud Storage client.

In [ ]:
try:
  # Initialize the client.
  client = storage.Client()
  # Get the bucket object using its name.
  gcs_bucket_obj = client.get_bucket(bucket)

  # The prefix for list_blobs should be the path relative to the bucket root.
  blob_prefix = f'{datasets_folder}/{dataset}/'
  blobs = gcs_bucket_obj.list_blobs(prefix=blob_prefix, match_glob='**/image_manifest.json')

  # Construct full GCS URIs from the blobs.
  manifest_uris = [f'gs://{bucket}/{blob.name}' for blob in blobs]

except Exception as e:
  print(f'Error listing GCS files: {e}')
  manifest_uris = []

print(f'Found {len(manifest_uris)} manifests for dataset {dataset}.')

### Load the manifests

Load each manifest into memory and update the asset name.

In [ ]:
manifests = []
for manifest_uri in manifest_uris:
  # Parse the manifest URI to get the file path within the bucket.
  parsed_uri = urlparse(manifest_uri)
  blob_path = parsed_uri.path.lstrip('/')
  # Download the manifest text and convert to a dictionary.
  blob = gcs_bucket_obj.blob(blob_path)
  json_content = blob.download_as_string()
  manifest_json = json.loads(json_content)
  # Update the Earth Engine asset name.
  asset_name = f'projects/{PROJECT}/assets/{dataset}/{manifest_json["properties"]["IMAGE_ID"]}'
  manifest_json['name'] = asset_name
  manifests.append(manifest_json)

In [ ]:
  # Print the first manifest to check.
  print(json.dumps(manifests[0], indent=2))

### Create the COG-backed `Image` assets

Iterate over the list of manifests and send each one to `ee.data.startExternalImageIngestion()`.

In [ ]:
for manifest in manifests:
  try:
    result = ee.data.startExternalImageIngestion(manifest, allow_overwrite=True)
    asset_id = result.get('name')
    print(f'Created asset: {asset_id}')
  except ee.EEException as e:
    print(f'Failed to create asset for {asset_name}: {e}')

### Check the results

Print the size of the collection and the first image for a sanity check.

In [ ]:
collection = ee.ImageCollection(f'projects/{PROJECT}/assets/{dataset}')
print('Collection size: ' + str(collection.size().getInfo()))
print(collection.first().getInfo())

## Unquantize the imagery, if necessary

The COGs are stored as signed 8-bit values (between -127 and 127 inclusive, as -128 is reserved as the "no data" value) in each channel of each pixel. If you need floats for further analysis, you may use a function like this to rescale to [-1,1].

In [ ]:
def unquantize_image(image):
  """
  Undoes the quantization of a Custom Satellite Embedding image.

  Args:
    image: A quantized Custom Satellite Embedding image. This will have raw
      signed 8-bit values (between -127 and 127 inclusive, as -128 is reserved
      as the "no data" value) in each channel of each pixel.

  Returns:
    An analysis-ready version of the image with floating-point values
    between -1 and 1.
  """
  # Mask the reserved no-data value -128 before unquantizing.
  masked = image.updateMask(image.neq(-128))
  # Re-scale pixels to be between (-1, 1) by undoing the int8 quantization.
  unquantized = masked.divide(127.5).pow(2).multiply(masked.signum()).float()
  # Copy regular and system metadata.
  return (unquantized
      .clip(image.geometry())
      .copyProperties(image)
      .copyProperties(image, ['system:time_start', 'system:time_end'])
  )

In [ ]:
print(unquantize_image(collection.first()).getInfo())

### Check the results visually with `geemap`

In [ ]:
Map = geemap.Map(center=(32.8287, -115.45131), zoom=13)
Map.addLayer(collection.map(unquantize_image).mosaic(), {'min': -0.36, 'max': 0.03}, 'rgb1')
Map

## Make dynamic collections using `ee.Image.loadGeoTiff()`

The previous section demonstrated how to make saved `Image` and `ImageCollection` assets backed by the COGs. The `ImageCollection` can be loaded directly in Earth Engine and used for further analysis.  This section demonstrates [`ee.Image.loadGeoTIFF`](https://developers.google.com/earth-engine/apidocs/ee-image-loadgeotiff) to make the collection dynamically.  Here we'll use the `manifest.txt` file in the dataset directory ([example](https://storage.mtls.cloud.google.com/alphaearth_foundations_custom/custom_satellite_embedding/v1/meloland_california_2026_crop_cycles/manifest.txt)) for simplicity.

In [ ]:
def read_manifest(manifest_gcs_uri):
  """Reads the manifest file with the GCS URIs of all the COG files of a dataset.

  Args:
    manifest_gcs_uri: The GCS URI of the manifest file.

  Returns:
    A list of GCS URIs (strings) of the COG files.
  """
  # Read the raw manifest from GCS.
  manifest_text = ee.Blob(manifest_gcs_uri).string().getInfo()
  raw_uri_list = manifest_text.split('\n')

  # Clean up the list (trim whitespace and remove blank lines).
  cleaned_uris = [uri.strip() for uri in raw_uri_list if uri.strip()]
  return cleaned_uris


def read_custom_embedding_image(cog_uri):
  """
  Creates an EE Image from the GCS URI of a Custom Satellite Embedding image.

  Args:
    cog_uri: A GCS URI of a Custom Satellite Embedding image, formatted as:
      'gs://{bucket}/{datasets_folder}/{dataset}/{start_date}_{end_date}/{cog_filename}.tiff'

  Returns:
    The image from GCS with properties parsed from the URI string.
  """
  # Parse field values from the URI.
  parts = cog_uri.split('/')
  dataset_name = parts[-4]
  time_interval_folder = parts[-3]
  time_interval = time_interval_folder.split('_')
  start_str = time_interval[0]
  end_str = time_interval[1]

  embedding_image = ee.Image.loadGeoTIFF(cog_uri)

  return embedding_image.set({
      'source_uri': cog_uri,
      'dataset': dataset_name,
      'system:time_start': ee.Date.parse('yyyyMMdd', start_str).millis(),
      'system:time_end': ee.Date.parse('yyyyMMdd', end_str).millis()
  })


def collection_from_custom_aef_tiffs(cog_gcs_uris):
  """
  Converts a list of Custom Satellite Embedding COG GCS URIs to an image collection.

  Args:
    cog_gcs_uris: A list of GCS URIs that link to Custom Satellite Embedding images.
      These URIs should be formatted as:
      'gs://{bucket}/{datasets_folder}/{dataset}/{start_date}_{end_date}/{cog_filename}.tiff'

  Returns:
    An image collection containing all the images from GCS
    with start/end dates parsed from the URI.
  """
  image_list = [read_custom_embedding_image(uri) for uri in cog_gcs_uris]
  return ee.ImageCollection(image_list)


def get_time_interval_str(image):
  """
  Returns the date interval of a Custom Satellite Embedding image as an EE string.

  Args:
    image: A Custom Satellite Embedding image.

  Returns:
    The start and end dates of the image as a single string.
    For example, an image with a start and end time of '2020-08-15' and
    '2020-09-15' returns the string '2020-08-15_2020-09-15'.
  """
  start_date_str = ee.Date(image.get('system:time_start')).format('YYYY-MM-DD')
  end_date_str = ee.Date(image.get('system:time_end')).format('YYYY-MM-DD')
  return start_date_str.cat('_').cat(end_date_str)

In [ ]:
manifest_uri = f'gs://{bucket}/{datasets_folder}/{dataset}/manifest.txt'
cog_paths = read_manifest(manifest_uri);
print('GeoTIFF Paths from the Manifest:', cog_paths);

quantized_image_coll = collection_from_custom_aef_tiffs(cog_paths);
quantized_image_coll = quantized_image_coll.map(
    lambda image: image.set('time_interval', get_time_interval_str(image)))

print('Quantized ImageCollection size:' + str(quantized_image_coll.size().getInfo()))
print('Quantized ImageCollection first:' + str(quantized_image_coll.first().getInfo()))

unquantized_image_coll = quantized_image_coll.map(unquantize_image)
print('Unquantized ImageCollection first:' + str(unquantized_image_coll.first().getInfo()))

In [ ]:
Map = geemap.Map(center=(32.8287, -115.45131), zoom=13)
Map.addLayer(unquantized_image_coll.mosaic(), {"min": -0.36, "max": 0.03}, "rgb1")
Map

# What's next?

Explore [other Custom Satellite Embeddings samples](https://developers.google.com/maps/documentation/custom-satellite-embeddings/samples).